<a href="https://colab.research.google.com/github/Phani-ISB/Reconciliation_Modules/blob/main/2_Bank_Reconciliation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import all required libraries

In [ ]:
pip install -q rapidfuzz

In [ ]:
import numpy as np
import pandas as pd
from datetime import timedelta
from rapidfuzz import fuzz

# Data Ingestion

In [ ]:
# Loading excel files of Ledger and Bank statement

ledger_df = pd.read_excel('/content/Ledger_InputData (1).xlsx')
bank_df = pd.read_excel('/content/Statement_InputData (1).xlsx')

In [ ]:
# Standardizing column names
ledger_df.columns = ledger_df.columns.str.strip()
bank_df.columns = bank_df.columns.str.strip()

print("Ledger Shape:", ledger_df.shape)
print("Bank Shape:", bank_df.shape)

Ledger Shape: (2977, 10)
Bank Shape: (250, 11)


# EDA

In [ ]:
ledger_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2977 entries, 0 to 2976
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Doc Type            2938 non-null   object        
 1   Posting Dt          2977 non-null   datetime64[ns]
 2   Doc No              2977 non-null   int64         
 3   Cheque No           14 non-null     float64       
 4   Description         1268 non-null   object        
 5   Particulars         832 non-null    object        
 6   Debit-Credit        2977 non-null   float64       
 7   Bank                2946 non-null   object        
 8   GL A/C              2969 non-null   float64       
 9   GL A/c Description  2977 non-null   object        
dtypes: datetime64[ns](1), float64(3), int64(1), object(5)
memory usage: 232.7+ KB


In [ ]:
bank_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Date          250 non-null    datetime64[ns]
 1   Unique ID     215 non-null    object        
 2   Bank Ref      249 non-null    object        
 3   Cust Ref      249 non-null    object        
 4   Narration     249 non-null    object        
 5   Narration 2   249 non-null    object        
 6   Opening Flag  1 non-null      object        
 7   Credit-Debit  250 non-null    float64       
 8   BANK          250 non-null    object        
 9   Credit        34 non-null     float64       
 10  Debit         215 non-null    float64       
dtypes: datetime64[ns](1), float64(3), object(7)
memory usage: 21.6+ KB


In [ ]:
# Create unified amount columns
ledger_df['Amount'] = ledger_df['Debit-Credit'].fillna(0)
bank_df['Amount'] = bank_df['Credit-Debit'].fillna(0)

In [ ]:
#  Understanding Debit/Credit in Ledger & Statement
print("\n Debits & Credits in LEDGER")
print("Ledger Debit Total:", ledger_df[ledger_df['Amount'] > 0]['Amount'].sum())
print("Ledger Credit Total:", ledger_df[ledger_df['Amount'] < 0]['Amount'].sum())
print("Ledger balances against Debit/Credit:", ledger_df['Amount'].sum())


print("\n Debits & Credits in STATEMENT")
print("Bank Debit Total:", bank_df[bank_df['Amount'] < 0]['Amount'].sum())
print("Bank Credit Total:", bank_df[bank_df['Amount'] > 0]['Amount'].sum())
print("Bank balances against Debit/Credit:", bank_df['Amount'].sum())

print("\n Overall Difference:", ledger_df['Amount'].sum() - bank_df['Amount'].sum())


 Debits & Credits in LEDGER
Ledger Debit Total: 116410182.16
Ledger Credit Total: -123096062.35779999
Ledger balances against Debit/Credit: -6685880.19780001

 Debits & Credits in STATEMENT
Bank Debit Total: -32271013.099999998
Bank Credit Total: 31409973.31
Bank balances against Debit/Credit: -861039.7899999991

 Overall Difference: -5824840.407800011


In [ ]:
# Normalize narration text (regex stripping)
ledger_df['Narration'] = ledger_df['Description'].astype(str).str.lower().str.strip()
bank_df['Narration'] = (bank_df['Narration'].astype(str) + " " + bank_df['Narration 2'].astype(str)).str.lower().str.strip()

In [ ]:
# Duplicates Check
print("Ledger Duplicates:", ledger_df.duplicated().sum())
print("Bank Duplicates:", bank_df.duplicated().sum())

Ledger Duplicates: 378
Bank Duplicates: 0


In [ ]:
# Duplicates data
duplicate_rows = ledger_df[ledger_df.duplicated(keep=False)]

# Sort them so the identical rows are next to each other
duplicate_rows.sort_values(by=list(ledger_df.columns)).head(10)

,Doc Type,Posting Dt,Doc No,Cheque No,Description,Particulars,Debit-Credit,Bank,GL A/C,GL A/c Description,Amount,Narration
1910,KZ,2021-01-12,601017,NaN,Salary,PAID AGAINST EE EXP REIMBURSE,-145.0,WELLS FARGO BANK-121000248,213002.0,ACCOUNTS PAYABLE - STAFF,-145.0,salary
1911,KZ,2021-01-12,601017,NaN,Salary,PAID AGAINST EE EXP REIMBURSE,-145.0,WELLS FARGO BANK-121000248,213002.0,ACCOUNTS PAYABLE - STAFF,-145.0,salary
33,KZ,2021-01-18,601038,NaN,Salary,PAID AGAINST EE EXPENSE REIMB,-250.0,WELLS FARGO BANK-121000248,213002.0,ACCOUNTS PAYABLE - STAFF,-250.0,salary
34,KZ,2021-01-18,601038,NaN,Salary,PAID AGAINST EE EXPENSE REIMB,-250.0,WELLS FARGO BANK-121000248,213002.0,ACCOUNTS PAYABLE - STAFF,-250.0,salary
35,KZ,2021-01-18,601038,NaN,Salary,PAID AGAINST EE EXPENSE REIMB,-250.0,WELLS FARGO BANK-121000248,213002.0,ACCOUNTS PAYABLE - STAFF,-250.0,salary
38,KZ,2021-01-18,601038,NaN,Salary,PAID AGAINST EE EXPENSE REIMB,-250.0,WELLS FARGO BANK-121000248,213002.0,ACCOUNTS PAYABLE - STAFF,-250.0,salary
1683,KZ,2021-01-18,601038,NaN,Salary,PAID AGAINST EE EXPENSE REIMB,-250.0,WELLS FARGO BANK-121000248,213002.0,ACCOUNTS PAYABLE - STAFF,-250.0,salary
30,KZ,2021-01-18,601038,NaN,Salary,PAID AGAINST EE EXPENSE REIMB,-150.0,WELLS FARGO BANK-121000248,213002.0,ACCOUNTS PAYABLE - STAFF,-150.0,salary
37,KZ,2021-01-18,601038,NaN,Salary,PAID AGAINST EE EXPENSE REIMB,-150.0,WELLS FARGO BANK-121000248,213002.0,ACCOUNTS PAYABLE - STAFF,-150.0,salary
848,KZ,2021-01-18,601038,NaN,Salary,PAID AGAINST EE EXPENSE REIMB,-150.0,WELLS FARGO BANK-121000248,213002.0,ACCOUNTS PAYABLE - STAFF,-150.0,salary


# Reconciliation by Rules

In [ ]:
ledger_df['Matched'] = False
bank_df['Matched'] = False

DATE_TOLERANCE =3
AMOUNT_TOLERANCE = 0.00
FUZZY_THRESHOLD = 50

all_matches = []


In [ ]:
# Helper functions for rules

def amount_match(a, b):
    return abs(a - b) <= AMOUNT_TOLERANCE

def date_exact(d1, d2):
    return d1 == d2

def date_range(d1, d2):
    return abs((d1 - d2).days) <= DATE_TOLERANCE

def narration_exact(n1, n2):
    return n1 == n2

def narration_fuzzy(n1, n2):
    if pd.isna(n1) or pd.isna(n2):
        return False
    return fuzz.partial_ratio(n1, n2) >= FUZZY_THRESHOLD

In [ ]:
# Defining function for matchig records

def match_records(rule_name, ledger_df, bank_df, condition_func):
    matches = []
  # Unmatched transactions are filtered
    ledger_unmatched = ledger_df[~ledger_df['Matched']]
    bank_unmatched = bank_df[~bank_df['Matched']]

    for li, lrow in ledger_unmatched.iterrows():
        # Prefiltering by Amount
        candidates = bank_unmatched[
            abs(bank_unmatched['Amount'] - lrow['Amount']) <= AMOUNT_TOLERANCE
        ]

        if candidates.empty:
            continue

        # Apply rule
        candidates = candidates[
            candidates.apply(lambda brow: condition_func(lrow, brow), axis=1)
        ]

        if not candidates.empty:
            bi = candidates.index[0]

            ledger_df.at[li, 'Matched'] = True
            bank_df.at[bi, 'Matched'] = True

            matches.append({
                "Ledger_Index": li,
                "Bank_Index": bi,
                "Rule": rule_name,
                "Ledger_Amount": lrow['Amount'],
                "Bank_Amount": bank_df.loc[bi, 'Amount']
            })

    return matches

In [ ]:
# Defining Rules in sequence

RECON_RULES = [
    ("Rule 1: Exact Narration + Exact Date",
     lambda l, b: narration_exact(l['Narration'], b['Narration']) and date_exact(l['Posting Dt'], b['Date'])),

    ("Rule 2: Exact Narration + Date Range",
     lambda l, b: narration_exact(l['Narration'], b['Narration']) and date_range(l['Posting Dt'], b['Date'])),

    ("Rule 3: Exact Narration Only",
     lambda l, b: narration_exact(l['Narration'], b['Narration'])),

    ("Rule 4: Fuzzy Narration + Exact Date",
     lambda l, b: narration_fuzzy(l['Narration'], b['Narration']) and date_exact(l['Posting Dt'], b['Date'])),

    ("Rule 5: Fuzzy Narration + Date Range",
     lambda l, b: narration_fuzzy(l['Narration'], b['Narration']) and date_range(l['Posting Dt'], b['Date'])),

    ("Rule 6: Just Exact Date",
     lambda l, b: date_exact(l['Posting Dt'], b['Date'])),

    ("Rule 7: Date Range Only",
     lambda l, b: date_range(l['Posting Dt'], b['Date']))
]

In [ ]:
# Execution Loop
all_matches = []

for rule_name, condition_func in RECON_RULES:
    new_matches = match_records(rule_name, ledger_df, bank_df, condition_func)
    all_matches.extend(new_matches)

In [ ]:
all_matches

In [ ]:
# Collect all the results
recon_df = pd.DataFrame(all_matches)

In [ ]:
recon_df

,Ledger_Index,Bank_Index,Rule,Ledger_Amount,Bank_Amount
0,10,55,Rule 4: Fuzzy Narration + Exact Date,-2065.10,-2065.10
1,11,168,Rule 4: Fuzzy Narration + Exact Date,-76.67,-76.67
2,103,140,Rule 4: Fuzzy Narration + Exact Date,-39295.40,-39295.40
3,104,141,Rule 4: Fuzzy Narration + Exact Date,-3867.89,-3867.89
4,105,142,Rule 4: Fuzzy Narration + Exact Date,-411.30,-411.30
...,...,...,...,...,...
159,2759,3,Rule 6: Just Exact Date,-9550.73,-9550.73
160,2760,4,Rule 6: Just Exact Date,-5120.00,-5120.00
161,2761,5,Rule 6: Just Exact Date,-2465.98,-2465.98
162,2762,6,Rule 6: Just Exact Date,-654.94,-654.94


In [ ]:
match_summary = recon_df['Rule'].value_counts().reset_index()
match_summary.columns = ['Rule','Count']

In [ ]:
print(match_summary)

                                   Rule  Count
0  Rule 4: Fuzzy Narration + Exact Date    123
1               Rule 6: Just Exact Date     39
2  Rule 5: Fuzzy Narration + Date Range      2


In [ ]:
# Filtering unmatched transactions in both ledger and bank statements
unmatched_ledger = ledger_df[~ledger_df['Matched']].copy()
unmatched_bank = bank_df[~bank_df['Matched']].copy()

In [ ]:
unmatched_bank.info()

<class 'pandas.core.frame.DataFrame'>
Index: 88 entries, 1 to 248
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Date          88 non-null     datetime64[ns]
 1   Unique ID     62 non-null     object        
 2   Bank Ref      87 non-null     object        
 3   Cust Ref      87 non-null     object        
 4   Narration     88 non-null     object        
 5   Narration 2   87 non-null     object        
 6   Opening Flag  1 non-null      object        
 7   Credit-Debit  88 non-null     float64       
 8   BANK          88 non-null     object        
 9   Credit        11 non-null     float64       
 10  Debit         76 non-null     float64       
 11  Amount        88 non-null     float64       
 12  Matched       88 non-null     bool          
dtypes: bool(1), datetime64[ns](1), float64(4), object(7)
memory usage: 9.0+ KB


In [ ]:
unmatched_ledger.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2813 entries, 0 to 2976
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Doc Type            2774 non-null   object        
 1   Posting Dt          2813 non-null   datetime64[ns]
 2   Doc No              2813 non-null   int64         
 3   Cheque No           14 non-null     float64       
 4   Description         1213 non-null   object        
 5   Particulars         676 non-null    object        
 6   Debit-Credit        2813 non-null   float64       
 7   Bank                2782 non-null   object        
 8   GL A/C              2805 non-null   float64       
 9   GL A/c Description  2813 non-null   object        
 10  Amount              2813 non-null   float64       
 11  Narration           2813 non-null   object        
 12  Matched             2813 non-null   bool          
dtypes: bool(1), datetime64[ns](1), float64(4), int64(1), 